# ForestGuard — 01 Extracción de datos
Obtención de registros oficiales desde la API ArcGIS de la Secretaría Nacional de Gestión de Riesgos.


## 1. Carga del dataset

Primero cargamos los datos oficiales de incendios forestales registrados en Ecuador durante 2025. La información se obtiene directamente desde el servicio de la Secretaría Nacional de Gestión de Riesgos y luego se convierte en un DataFrame de pandas para poder trabajar con ella.


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

URL = "https://sgrportal.gestionderiesgos.gob.ec/server/rest/services/Hosted/incendios_forestales_2025_shp/FeatureServer/0/query"

def descargar_datos_arcgis(url, page_size=2000):
    registros = []
    offset = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "false",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "json"
        }

        respuesta = requests.get(url, params=params, timeout=60)
        respuesta.raise_for_status()
        datos = respuesta.json()

        if "error" in datos:
            raise RuntimeError(datos["error"])

        features = datos.get("features", [])
        if not features:
            break

        registros.extend([f["attributes"] for f in features])

        if len(features) < page_size:
            break

        offset += page_size

    return pd.DataFrame(registros)

df = descargar_datos_arcgis(URL)
df_original = df.copy()

columnas_vista = [
    "provincia", "canton", "parroquia", "mes",
    "latitud", "longitud", "hadecobert", "nivelinf", "region"
]
columnas_vista = [c for c in columnas_vista if c in df.columns]

df[columnas_vista].head()


In [ ]:
from pathlib import Path
ruta_raw = Path("data/raw")
ruta_raw.mkdir(parents=True, exist_ok=True)
archivo_raw = ruta_raw / "incendios_forestales_2025_raw.csv"
df.to_csv(archivo_raw, index=False, encoding="utf-8-sig")
print(f"Dataset original guardado en: {archivo_raw}")
print("Dimensiones:", df.shape)
